# SIMC 2026 — Endeavour Challenge: combined solution notebook

This notebook stitches together the code for all parts of the challenge that have a
computational component, in solution order, so the whole pipeline runs top-to-bottom from
one kernel:

- **Task 1 — Markov-matrix model** (numerical validation, parts f & h). Self-contained;
  reads no external data.
- **Task 2 — Power Method & SVD** is **analytical only** (proofs of convergence, the
  spectral-gap rate, the X^T X / X X^T equivalence, and a worked 3×2 SVD by hand). It has
  **no code**; the full worked solution is in the preliminary report. A short marker cell
  sits where Task 2 would go.
- **Task 3 — Design / covariance / Gram matrices** (PCA & clustering, parts a–g). Reads
  `data/input/mystery_matrix1.npy`; writes figures to `data/output/TASK 3/`.

**How to run.** Execute cells in order. Task 3 auto-locates the repo root, so the notebook
works whether launched from `src/` or the repo root. The canonical, individually-maintained
notebooks remain `src/task1/task1.ipynb` and `src/task_3.ipynb`; this file is the merged view.


# Task 1 — Numerical validation of the Markov-matrix model

**What Task 1 is.** Two organism types have fractional populations $p_1(t),p_2(t)$ evolving by
$\vec{P}(t+1)=\mathbb{M}\,\vec{P}(t)$ with the column-stochastic, lower-triangular matrix
$\mathbb{M}=\bigl(\begin{smallmatrix}1-\epsilon & 0\\ \epsilon & 1\end{smallmatrix}\bigr)$.
Parts (a)–(e) and (g) are **analytical** (eigenvalues $\lambda_1=1,\ \lambda_2=1-\epsilon$;
eigenvectors $\vec{\nu}_1=(0,1)^\top,\ \vec{\nu}_2=(1,-1)^\top$; closed form
$\vec{P}(t)=c_1\vec{\nu}_1+c_2\lambda_2^{\,t}\vec{\nu}_2$) and are written up in full in the
preliminary report. **This notebook is the computational half — parts (f) and (h)** — the
numerical validation the handwritten notes had left unattempted.

**What this notebook does.** For each closed form it runs a *two-track* check:
1. **Direct simulation** — iterate the recursion $\vec{P}(t+1)=\mathbb{M}\,\vec{P}(t)$ with no
   analytical input (the ground truth).
2. **Closed form** — evaluate the eigenmode expression independently.
3. **Assert agreement** to machine precision ($<10^{-10}$), so any future edit that breaks a
   formula fails loudly rather than silently producing a wrong figure.

**Data contract.** No external data is read. The only inputs are the chosen parameters
($\epsilon=0.1$ for 1f; $\epsilon_0=0.5,\ t_0=10$ for 1h) and the horizon $T=100$. Nothing is
random, so every run is bit-for-bit reproducible. Two figures are written to `data/output/`
(`task1f_trajectories.png`, `task1h_trajectories.png`) for the report.

**Cell map.** (1f) validate+plot constant $\epsilon$ → (1h) validate+plot time-varying
$\epsilon(t)$. The 1h cells reuse `t`, `c1`, `c2`, `v1`, `v2`, `P_num` defined in the 1f cells,
so the notebook must be run top-to-bottom.

## Part 1f — Validation of parts (d) and (e)

**Goal.** Validate the closed-form expression for $\vec{P}(t)$ from part (d) and the asymptotic
limit from part (e).

**Strategy.** Simulate the recursion $\vec{P}(t+1) = \mathbb{M}\,\vec{P}(t)$ directly, then compute
$\vec{P}(t)$ independently from the eigenmode decomposition (parts 1b–1c), and assert the two agree
to machine precision. The asymptotic check then confirms $\vec{P}(T) \to \vec{\nu}_1 = (0,1)^\top$.

In [ ]:
# ===========================================================
# Part 1f — numerical validation of parts (d) and (e)
# ===========================================================
'''Purpose. Confirm the closed-form solution P(t) = c1*v1 + c2*(1-eps)^t * v2 (part 1d)
and its limit P(t) -> (0,1) (part 1e) by checking them against a brute-force iteration of
the recursion that uses no analytical input.

Method (two independent tracks):
  - Track A (ground truth): iterate P(t+1) = M @ P(t) for T steps.
  - Track B (closed form): decompose P(0) in the eigenbasis {v1, v2}, then evaluate the
    scalar-power formula for all t at once via broadcasting.
  The eigenpairs are taken from part 1b (lambda1=1 persistent, lambda2=1-eps decaying); the
  coefficients c1, c2 are obtained by solving the 2x2 linear system P(0) = c1*v1 + c2*v2,
  not hard-coded, so the cell also re-derives the c1=1, c2=1/2 of part 1c.

Validation & outputs. max|A - B| must be < 1e-10 (asserted, so a broken formula fails
loudly). P(T) is printed to show convergence to (0,1). Defines t, c1, c2, v1, v2, P_num,
which the 1h cells reuse — run this cell first.'''
import numpy as np

# --- Parameters ---
eps = 0.1                                 # conversion rate per generation
p0  = np.array([0.5, 0.5])                # initial state (p2 = 1 - p1)
T   = 100                                 # generations to simulate

# Markov matrix; lower-triangular, column-stochastic.
M = np.array([[1 - eps, 0.0],
              [eps,     1.0]])

# --- Stage 1: direct simulation ---
# Ground truth: iterate the recursion with no analytical input.
P_num = np.zeros((T + 1, 2))
P_num[0] = p0
for t_step in range(T):
    P_num[t_step + 1] = M @ P_num[t_step]

# --- Stage 2: closed form via eigenmode decomposition ---
# Eigenvectors from 1b (lambda_1 = 1, lambda_2 = 1 - eps).
v1 = np.array([0,  1])                    # persistent mode
v2 = np.array([1, -1])                    # decaying   mode

# Solve P(0) = c1*v1 + c2*v2 for the coefficients in the eigenbasis.
c1, c2 = np.linalg.solve(np.column_stack([v1, v2]), p0)
print(f"c1 = {c1},  c2 = {c2}")           # expect c1 = 1, c2 = 0.5

# Evaluate the closed form for all t at once via broadcasting.
t = np.arange(T + 1)
P_closed = c1 * v1 + c2 * ((1 - eps) ** t)[:, None] * v2

# --- Stage 3: validation ---
# Both arrays must agree to machine precision; the assertion catches any
# future edit that breaks the closed form.
max_err = np.max(np.abs(P_num - P_closed))
print(f"max |numeric - closed form| = {max_err:.2e}")
assert max_err < 1e-10, "closed form disagrees with simulation -- check 1d"

# Asymptotic check (validates 1e): P(T) sits near [0, 1] = nu_1.
print(f"P(T={T}) = {P_num[-1]}   (should approach [0, 1])")

### Part 1f — Visualisation

Two-panel figure for the report:

- **Panel A (linear scale).** Trajectories $p_1(t), p_2(t)$ converging to $(0, 1)$. Visual confirmation of part (e); dotted horizontal lines mark the asymptotes.
- **Panel B (log scale).** Numeric $p_1(t)$ overlaid on the closed form $\tfrac{1}{2}(1-\epsilon)^t$. The straight line on semi-log $y$ axes proves the decay is geometric with rate $1-\epsilon$ (validates the rate); numeric dots sitting on top of the line validate the closed form from (d).


In [ ]:
# ===== Shared plotting setup (used by both 1f and 1h) =====
# Notebook-friendly Matplotlib setup: prefer inline; fallback to Agg if unavailable
try:
    get_ipython().run_line_magic('matplotlib', 'inline')
except Exception:
    import matplotlib
    try:
        matplotlib.use('Agg')
    except Exception:
        pass
import matplotlib
print('matplotlib', matplotlib.__version__, 'backend', matplotlib.get_backend())

In [ ]:
# ===== Part 1f — plot (linear + log) =====
import matplotlib.pyplot as plt
from pathlib import Path

# Match the report's serif typesetting.
plt.rcParams.update({"font.family": "serif", "font.size": 11})

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# --- Panel A: linear-scale trajectories ---
# Dotted horizontal lines mark the asymptotic values, making the limit
# in 1e visible at a glance.
ax = axes[0]
ax.plot(t, P_num[:, 0], "o-", color="tab:blue",   ms=3, lw=1,
        label=r"$p_1(t)$ (type 1)")
ax.plot(t, P_num[:, 1], "o-", color="tab:orange", ms=3, lw=1,
        label=r"$p_2(t)$ (type 2)")
ax.axhline(0, color="tab:blue",   ls=":", lw=1, alpha=0.6)
ax.axhline(1, color="tab:orange", ls=":", lw=1, alpha=0.6)
ax.set_xlabel(r"generation $t$")
ax.set_ylabel("fractional population")
ax.set_title(rf"Population trajectories ($\epsilon = {eps}$)")
ax.set_ylim(-0.02, 1.05)
ax.legend(loc="center right")
ax.grid(alpha=0.3)

# --- Panel B: log-scale; numeric vs closed form ---
# A straight line on semilog-y proves the decay is geometric with rate
# (1 - eps); numeric dots sitting on the line validate the closed form.
theory_p1 = 0.5 * (1 - eps) ** t          # (1/2)(1-eps)^t

ax = axes[1]
ax.semilogy(t, theory_p1,   "-",  color="black",    lw=1.2,
            label=r"theory: $\frac{1}{2}(1-\epsilon)^t$")
ax.semilogy(t, P_num[:, 0], "o",  color="tab:blue", ms=4,
            label=r"numeric $p_1(t)$")
ax.set_xlabel(r"generation $t$")
ax.set_ylabel(r"$p_1(t)$  (log scale)")
ax.set_title("Geometric decay of type-1: numeric vs closed form")
ax.legend()
ax.grid(which="both", alpha=0.3)

fig.tight_layout()

# Save under data/output/ (repo convention). Walk up to the repo root so
# the path is correct regardless of the kernel's working directory.
root = Path.cwd().resolve()
while not (root / "data").is_dir() and root != root.parent:
    root = root.parent
out_dir = root / "data" / "output"
out_dir.mkdir(parents=True, exist_ok=True)
fig.savefig(out_dir / "task1f_trajectories.png", dpi=150, bbox_inches="tight")
plt.show()


## Part 1h — Numerical validation of part (g): time-varying $\epsilon(t)$

The conversion rate now decays, $\epsilon(t) = \epsilon_0\,e^{-t/t_0}$, so the matrix $\mathbb{M}(t)$ changes every generation. The eigenvectors stay $\epsilon$-independent (same $\vec{\nu}_1, \vec{\nu}_2$ as 1b); only $\lambda_2(t) = 1-\epsilon(t)$ varies, so the geometric factor $(1-\epsilon)^t$ from 1d becomes a **cumulative product** $\Pi(t) = \prod_{s=0}^{t-1}\bigl(1-\epsilon(s)\bigr)$.

**Strategy.** Same two-track check as 1f — direct simulation with a per-step matrix $\mathbb{M}(t)$ vs the closed form $\vec{P}(t) = c_1\vec{\nu}_1 + c_2\,\Pi(t)\,\vec{\nu}_2$ — then plot the $\epsilon(t)$ trajectories **on top of the constant-$\epsilon$ curves from 1f** for comparison. Headline result (1g): because $\sum_s \epsilon(s)$ is finite, $p_1$ no longer vanishes but **freezes** at a positive residue $\tfrac{1}{2}\Pi(\infty) \approx \tfrac{1}{2}e^{-\epsilon_0 t_0}$.


In [ ]:
# ===========================================================
# Part 1h — numerical validation of part (g): time-varying eps(t)
# ===========================================================
'''Purpose. Validate the part-(g) generalisation to a decaying rate eps(t)=eps0*exp(-t/t0),
and its qualitative consequence: type-1 is NOT fully depleted but freezes at a positive
residue. This is the key correction over the notes, which had (1-eps(t))^t instead of the
cumulative product.

Why a product, not a power. With eps time-varying the update matrix M(t) changes every step,
but its eigenvectors v1, v2 are eps-independent (same as 1b) and only lambda2(t)=1-eps(t)
varies. Acting on P(0)=c1*v1+c2*v2, the v2 mode collects ONE factor (1-eps(s)) per step, so
the geometric power (1-eps)^t of 1d becomes the cumulative product Pi(t)=prod_{s<t}(1-eps(s)).

Method (mirrors 1f). Track A: iterate with a freshly built M(s) each step. Track B: the
closed form c1*v1 + c2*Pi(t)*v2, with Pi computed by np.cumprod (Pi(0)=1, the empty product).
Reuses t, c1, c2, v1, v2 from the 1f cell.

Validation & outputs. max|A - B| < 1e-10 (asserted). Because sum_s eps(s) < infinity, Pi(inf)
is strictly positive, so p1 plateaus at c2*Pi(inf); we print this residue alongside the
small-eps / large-t0 estimate (1/2)e^(-eps0*t0) for a sanity check. Defines P_num_g, Pi,
plateau_p1 for the 1h plot cell.'''

eps0  = 0.5                               # initial conversion rate
t0    = 10.0                              # decay timescale of eps(t)
eps_t = eps0 * np.exp(-t / t0)            # eps(s) for s = 0..T  (t from 1f cell)

# --- Stage 1: direct simulation with a time-varying matrix ---
P_num_g = np.zeros((T + 1, 2))
P_num_g[0] = p0
for s in range(T):
    e = eps_t[s]
    M_s = np.array([[1 - e, 0.0],
                    [e,     1.0]])
    P_num_g[s + 1] = M_s @ P_num_g[s]

# --- Stage 2: closed form via the cumulative product Pi(t) ---
# Pi(0) = 1 (empty product); Pi(t) = prod_{s=0}^{t-1} (1 - eps(s)).
Pi = np.concatenate(([1.0], np.cumprod(1 - eps_t[:-1])))
P_closed_g = c1 * v1 + c2 * Pi[:, None] * v2   # reuse c1=1, c2=0.5, v1, v2 from 1f

# --- Stage 3: validation ---
max_err_g = np.max(np.abs(P_num_g - P_closed_g))
print(f"max |numeric - closed form| (1h) = {max_err_g:.2e}")
assert max_err_g < 1e-10, "1g closed form disagrees with simulation -- check the product"

# --- Asymptotic check (validates 1g): a frozen, non-zero residue ---
plateau_p1     = 0.5 * Pi[-1]             # actual residue p1(inf) = c2 * Pi(inf)
plateau_approx = 0.5 * np.exp(-eps0 * t0) # guide's small-eps / large-t0 estimate
print(f"P(T={T}) = {P_num_g[-1]}   (p1 freezes above 0, unlike 1f)")
print(f"residue p1(inf) = {plateau_p1:.6f}   "
      f"(approx (1/2)e^(-eps0*t0) = {plateau_approx:.6f})")

In [ ]:
# ===== Part 1h — plot: time-varying eps(t) vs constant eps (1f) =====
import matplotlib.pyplot as plt
from pathlib import Path

fig2, axes2 = plt.subplots(1, 2, figsize=(11, 4))

# --- Panel A: linear scale, both scenarios overlaid ---
# Faded = constant eps (1f, for reference); solid = time-varying eps(t) (1h).
ax = axes2[0]
ax.plot(t, P_num[:, 0],   "-",  color="tab:blue",   lw=1, alpha=0.35,
        label=r"$p_1$, const $\epsilon$ (1f)")
ax.plot(t, P_num[:, 1],   "-",  color="tab:orange", lw=1, alpha=0.35,
        label=r"$p_2$, const $\epsilon$ (1f)")
ax.plot(t, P_num_g[:, 0], "o-", color="tab:blue",   ms=3, lw=1.2,
        label=r"$p_1(t)$, $\epsilon(t)$ (1h)")
ax.plot(t, P_num_g[:, 1], "o-", color="tab:orange", ms=3, lw=1.2,
        label=r"$p_2(t)$, $\epsilon(t)$ (1h)")
ax.axhline(plateau_p1,     color="tab:blue",   ls=":", lw=1, alpha=0.7)
ax.axhline(1 - plateau_p1, color="tab:orange", ls=":", lw=1, alpha=0.7)
ax.set_xlabel(r"generation $t$")
ax.set_ylabel("fractional population")
ax.set_title(rf"Time-varying vs constant $\epsilon$  ($\epsilon_0={eps0},\ t_0={t0:g}$)")
ax.set_ylim(-0.02, 1.05)
ax.legend(loc="center right", fontsize=8)
ax.grid(alpha=0.3)

# --- Panel B: log scale -- the headline contrast ---
# Constant eps: p1 -> 0 (straight line on semilog-y). Time-varying eps(t):
# p1 *plateaus* at the frozen residue once eps(t) effectively switches off.
ax = axes2[1]
ax.semilogy(t, P_num[:, 0],   "-",  color="gray",     lw=1.2,
            label=r"const $\epsilon$: $p_1 \to 0$")
ax.semilogy(t, P_num_g[:, 0], "o",  color="tab:blue", ms=4,
            label=r"$\epsilon(t)$: $p_1(t)$")
ax.axhline(plateau_p1, color="tab:red", ls="--", lw=1,
           label=r"frozen residue $p_1(\infty)=\frac{1}{2}\Pi(\infty)$")
ax.set_xlabel(r"generation $t$")
ax.set_ylabel(r"$p_1(t)$  (log scale)")
ax.set_title(r"Decaying $\epsilon(t)$ freezes $p_1$ at a positive residue")
ax.legend(fontsize=8)
ax.grid(which="both", alpha=0.3)

fig2.tight_layout()

# Save under data/output/ (same repo-root resolution as the 1f cell).
root = Path.cwd().resolve()
while not (root / "data").is_dir() and root != root.parent:
    root = root.parent
out_dir = root / "data" / "output"
out_dir.mkdir(parents=True, exist_ok=True)
fig2.savefig(out_dir / "task1h_trajectories.png", dpi=150, bbox_inches="tight")
plt.show()


# Task 2 — The Power Method and Singular Value Decomposition  *(analytical — no code)*

Task 2 is solved entirely on paper; there is no notebook code to run. For completeness the
results proved in the report are:

1. **(a)** The Power Method $x_{k+1}=Xx_k/\lVert Xx_k\rVert$ converges to $\pm v_1$ whenever
   $x_0$ has a nonzero component along $v_1$ and $|\lambda_1|>|\lambda_2|$.
2. **(b)** Convergence is geometric at rate $|\lambda_2/\lambda_1|$ (the spectral gap).
3. **(c)** A vanishing gap ($|\lambda_1|=|\lambda_2|$) is a structural obstruction — the
   iterate oscillates or drifts in a degenerate eigenspace; near-degeneracy compounds with
   $10^{-16}$ round-off.
4. **(d)** $X^\top X$ and $XX^\top$ are symmetric PSD and share their nonzero eigenvalues
   $\sigma_i^2$ — the fact Task 3(f)/(g) rely on to connect the covariance and Gram views.
5. **(e)** Power-Method SVD algorithm (work with the smaller of $X^\top X$, $XX^\top$) plus a
   worked $3\times2$ example.

See **§2 of the preliminary report** for the full proofs.


In [ ]:
#Module purpose. Question 3a, part (i): detect and remove the corrupted "empty vial" measurements from the spectroscopy design matrix mystery_matrix1.npy.

'''Data contract / assumptions.
  - The file loads as a float64 array of shape (3401, 1000). The problem defines the design matrix X with samples as rows (N = 1000 vials) and features as
  columns (frequencies). The on-disk array is therefore transposed relative to that convention: its 1000 axis is the samples. We transpose once, up front,
  so that downstream code follows the report's row = sample convention. (Note: the problem text states M = 3041 features; the actual file has 3401 columns.
  We use the array's true dimension and flag the discrepancy rather than hard-coding either number.)
  - All entries are non-negative intensities in [0, ~0.85]; there are no NaN/inf values. This is asserted, not assumed silently.

  Detection method. Each measurement is reduced to a single scalar, its row norm ‖xᵢ‖₂ = total signal energy across all frequencies. A real sample has
  absorption peaks (high norm); an empty vial is low background noise everywhere (low norm). The norm distribution is bimodal with an empty gap, so a hard
  threshold placed anywhere inside the gap separates the two populations exactly.

  Threshold choice & robustness. The cutoff is 3.0, but the result is invariant to any threshold in [2.0, 4.0] because the gap is genuinely empty. This
  invariance is checked explicitly — it is the evidence that the cut is principled rather than arbitrary.

  Validation strategy. Two independent checks: (1) summary statistics (mean/max/per-channel spread) must show the removed group is flat and low while the
  kept group has structure; (2) a visual overlay of removed vs kept spectra must show flat noise vs clear peaks.

  Outputs. T (number removed, expected 40), a boolean mask empty, the cleaned matrix S_clean (960 × 3401), and a figure q3a_part_i.png. The cleaned matrix
  is the input to 3b onward.'''



In [ ]:
#Cell 1 — Imports

import os
import numpy as np
import matplotlib.pyplot as plt

# Run from the repository root so the relative data/ and data/output/TASK 3/ paths
# below resolve, whether this notebook is launched from src/ or the repo root.
for _ in range(5):
    if os.path.exists("data/input/mystery_matrix1.npy"):
        break
    os.chdir("..")
DATA_PATH = "data/input/mystery_matrix1.npy"


In [ ]:
#Cell 2 — Load and sanity-check the raw array

# Load the raw design-matrix file.
# NOTE: the file is stored as (3401, 1000). The problem convention is
# samples-as-rows with N = 1000 vials, so the file is effectively X^T.
raw = np.load(DATA_PATH)

print("raw shape :", raw.shape)
print("dtype     :", raw.dtype)
print("min / max :", raw.min(), raw.max())
print("NaN / inf :", np.isnan(raw).any(), np.isinf(raw).any())

# The problem states M = 3041 features; the file has 3401. We trust the file
# and key off the axis whose length is 1000 (the samples).
assert 1000 in raw.shape, "expected one axis to be the 1000 samples"



In [ ]:
#Cell 3 — Orient to "samples as rows"

# Transpose if necessary so that rows = samples (vials), cols = frequencies.
S = raw.T if raw.shape[0] != 1000 else raw
N, M = S.shape
print(f"design matrix oriented: {N} samples (rows) x {M} frequencies (cols)")



In [ ]:
#Cell 4 — Reduce each measurement to its row norm

# Row norm = total signal energy of one vial across all frequencies.
# Empty vials (background noise only) have small norm; real samples have peaks.
norms = np.linalg.norm(S, axis=1)
print("row-norm summary")
print("  min   :", norms.min())
print("  max   :", norms.max())
print("  mean  :", norms.mean())
print("  median:", np.median(norms))



In [ ]:
#Cell 5 — Inspect the distribution (find the gap)
# A text histogram makes the bimodal structure and the empty gap obvious.
hist, edges = np.histogram(norms, bins=40)
for h, e in zip(hist, edges[:-1]):
  print(f"  {e:5.2f} : {'#' * h} {h}")
# A tight clump of 40 sits at ~1.58-1.76, then an empty gap until ~4.15.



In [ ]:
#Cell 6 — Threshold and confirm robustness
#The empty-vial clump is well below the real samples. Any threshold inside the
#gap gives the same count, which is the evidence the cut is not arbitrary.
for thr in (2.0, 3.0, 4.0):
    print(f"  rows with norm < {thr}: {(norms < thr).sum()}")  # all 40

THRESHOLD = 3.0
empty = norms < THRESHOLD          # boolean mask of corrupted measurements
real  = ~empty

T = int(empty.sum())
print(f"\nT = {T} corrupted (empty-vial) measurements")
print("their norm range:", norms[empty].min(), "->", norms[empty].max())
print("remaining measurements:", int(real.sum()))   # 960



In [ ]:
#Cell 7 — Validate the removal (statistics)
# Removed group should be flat & low; kept group should have structure (peaks).
def describe(mask, label):
  """Print summary statistics (per-channel mean, max, mean per-row std) for the rows of S picked out by the boolean `mask`, tagged with `label`; used to contrast the removed (flat, low) vials with the kept (peaked) ones."""
  print(f"{label:12s}: mean/chan = {S[mask].mean():.4f}  "
      f"max = {S[mask].max():.4f}  "
      f"per-channel std = {S[mask].std(axis=1).mean():.4f}")
describe(empty, "empty (rm)")
describe(real,  "real (keep)")



In [ ]:
#Cell 8 — Validate the removal (figure)

# Single panel: the row-norm distribution with the empty-vial threshold marked.
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.hist(norms, bins=60, color="steelblue", edgecolor="k", linewidth=0.3)
ax.axvline(THRESHOLD, color="red", ls="--", label=f"threshold = {THRESHOLD}")
ax.set_xlabel(r"row norm  $\|x_i\|$")
ax.set_ylabel("count")
ax.set_title(f"Row-norm distribution (1000 measurements)\n"
             f"{T} empty vials isolated below the gap")
ax.legend()
plt.tight_layout()
import os
os.makedirs("data/output/TASK 3", exist_ok=True)
plt.savefig("data/output/TASK 3/q3a_part_i.png", dpi=120)
plt.show()

In [ ]:
#Cell 9 — Produce the cleaned matrix for downstream parts

  # Cleaned design matrix after removing the T empty-vial measurements.
  # This (960 x 3401) array is the input to 3a(ii) and 3b onward.
S_clean = S[real]
print("cleaned design matrix:", S_clean.shape)   # (960, 3401)

In [ ]:
#Module purpose. Question 3a, part (ii): find the "other error" beyond the empty vials.

'''The other error -- high-norm outliers.
  Inputs. the 1000 row norms and the empty-vial mask from part (i).
  Method. Sort the norms and inspect the UPPER tail. The largest gap above the median isolates a
    second, detached group of 36 vials (norm ~8.9), separated from the bulk by a gap about five
    times wider than any internal gap and forming the contiguous block of rows 924-959.
  Honest caveat. Such a coherent group could be a genuinely strong concoction rather than an
    error, so we (a) gather independent evidence in the next cell (it forms one isolated cluster
    whose removal drops the cluster count by exactly one), and (b) keep the removal reversible.
  Outputs. boolean mask `high`; the fully cleaned matrix S_clean (924 x 3401).'''

In [ ]:
#Cell 10 — Question 3a, part (ii): the other error (high-norm outliers)

# Besides the empty vials, the row-norm distribution shows a SECOND group far
# above the bulk, separated by a clear gap (~5x wider than any gap inside the
# bulk). These measurements form a contiguous block (rows 924-959) sitting
# directly in front of the empty vials (rows 960-999): the whole tail of the
# matrix is anomalous, so we flag them as the "other error".

sn   = np.sort(norms)
gaps = np.diff(sn)
mid  = len(sn) // 2
hi   = mid + int(np.argmax(gaps[mid:]))        # widest gap ABOVE the median
high_thr = 0.5 * (sn[hi] + sn[hi + 1])

high = norms > high_thr                         # mask: high-norm outliers
print(f"high-norm threshold : {high_thr:.3f}")
print(f"outliers found      : {int(high.sum())}  "
      f"(rows {np.where(high)[0].min()}-{np.where(high)[0].max()})")
print(f"norm range          : {norms[high].min():.2f}-{norms[high].max():.2f}"
      f"  (bulk median {np.median(norms):.2f})")

In [ ]:
#Cell 11 — Produce the fully cleaned matrix (both errors removed)

# Keep a vial only if it is neither an empty blank nor a high-norm outlier.
keep = real & ~high
S_clean = S[keep]                               # overwrite: this is now final

print("removed - empty vials    :", int(empty.sum()))   # 40
print("removed - high-norm group:", int(high.sum()))    # 36
print("cleaned design matrix    :", S_clean.shape)       # (924, 3401)

In [ ]:
#Cell 12 — Evidence: are the 36 high-norm vials their own cluster?

# Concrete test of the part-(ii) decision. Cluster the empties-removed data in
# PCA space WITH vs WITHOUT the 36 high-norm vials. If they are a single
# distinct group, removing them should drop the cluster count by exactly one
# and leave every other cluster untouched.
import os

def pca_scores(X, k=4):
    """Top-k principal-component scores of X via its Gram matrix (X-mean)(X-mean)^T -- cheaper than the covariance when features outnumber samples. Returns an (n_samples, k) array; column j is the score on principal component j."""
    # project samples onto their top-k principal components (via the Gram matrix)
    Xc = X - X.mean(0)
    w, v = np.linalg.eigh(Xc @ Xc.T)
    idx = np.argsort(w)[::-1][:k]
    return v[:, idx] * np.sqrt(np.maximum(w[idx], 0.0))

def count_clusters(P, r):
    """Leader clustering (numpy-only, no sklearn): scan the points P and start a new cluster centre whenever a point lies farther than radius r from every existing centre. Returns the number of clusters found."""
    # leader clustering (numpy-only): start a new cluster when a point is
    # farther than r from every existing cluster centre
    centres = [P[0]]
    for p in P[1:]:
        if np.sqrt(((np.array(centres) - p) ** 2).sum(1)).min() > r:
            centres.append(p)
    return len(centres)

for label, mask in [("WITH    high-norm (960 vials)", real),
                    ("WITHOUT high-norm (924 vials)", real & ~high)]:
    P = pca_scores(S[mask], k=4)
    D = np.sqrt(((P[:, None, :] - P[None, :, :]) ** 2).sum(-1))
    np.fill_diagonal(D, np.inf)
    r = 8 * np.median(D.min(1))                  # radius ~ 8x typical neighbour spacing
    print(f"{label}: {count_clusters(P, r)} clusters")

# How isolated is the group, in the top-4 PC space?
P = pca_scores(S[real], k=4)
is_high = high[real]                             # which of the 960 kept vials are high-norm
ch     = P[is_high].mean(0)
radius = np.sqrt(((P[is_high]  - ch) ** 2).sum(1)).mean()
gap    = np.sqrt(((P[~is_high] - ch) ** 2).sum(1)).min()
print(f"\nhigh-norm cluster radius = {radius:.2f}, gap to nearest other vial = {gap:.2f} "
      f"({gap / radius:.0f}x its own radius)")
print("=> removing the 36 deletes exactly one well-separated cluster; the rest are untouched.")

# Visual: top-2 PCs with the 36 highlighted
plt.figure(figsize=(6, 5))
plt.scatter(P[~is_high, 0], P[~is_high, 1], s=10, alpha=0.4, label="other vials")
plt.scatter(P[is_high, 0],  P[is_high, 1],  s=30, color="red", label="high-norm group (36)")
plt.xlabel("PC1"); plt.ylabel("PC2"); plt.legend()
plt.title("The 36 high-norm vials form one isolated cluster")
plt.tight_layout()
os.makedirs("data/output/TASK 3", exist_ok=True)
plt.savefig("data/output/TASK 3/q3a_part_ii_clusters.png", dpi=120)
plt.show()

In [ ]:
#Module purpose. Question 3b: per-feature variance and the covariance matrix.

'''Variance, mean-centering and the covariance matrix.
  Inputs. the cleaned design matrix S_clean (924 x 3401) from 3a.
  Method. Mean-center each feature (subtract its column mean) to get X_centered = X~. The
    per-feature variance is the diagonal of the covariance matrix C = (1/N~) X~^T X~, so we read
    it off directly as X_centered.var(axis=0) without ever forming the full 3401 x 3401 matrix.
    Convention: np.var divides by N~ (population variance), matching the definition of C.
  Sanity check. trace(C) = sum of variances = total variance (~7.90) = sum of eigenvalues (3c).
  Outputs. X_centered, N_tilde, the `variances` vector, j_star (=910); figure q3b_variance.png.'''

In [ ]:
#Cell 13 — Question 3b: mean-centre and per-feature variance

# Mean-centre each feature (column) so it has zero average across the N~ vials.
N_tilde = S_clean.shape[0]
X_centered = S_clean - S_clean.mean(axis=0)          # X~, shape (924, 3401)

# Covariance matrix C = (1/N~) X~^T X~  (M x M).  We only need its DIAGONAL here:
# C_jj = variance of feature j.  np.var divides by N~, matching the definition.
variances = X_centered.var(axis=0)                   # = diag(C)

# Sanity check: trace(C) = total variance (invariant under rotation/compression).
print("total variance  tr(C) =", round(variances.sum(), 4))

# 3b(i): the single feature with the largest variance
j_star = int(np.argmax(variances))
print(f"3b(i)  largest-variance feature  j* = {j_star}  "
      f"(variance = {variances[j_star]:.5f})")

In [ ]:
#Cell 14 — Question 3b(ii): plot the variance as a function of feature index

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(variances, lw=0.6, color="steelblue")
ax.axvline(j_star, color="red", ls="--", lw=1, label=f"j* = {j_star}  (largest-variance feature)")
ax.set_xlabel("feature index j  (frequency channel)")
ax.set_ylabel(r"variance  $C_{jj}$")
ax.set_title("Per-feature variance across the 3401 frequencies")
ax.legend()
plt.tight_layout()
os.makedirs("data/output/TASK 3", exist_ok=True)
plt.savefig("data/output/TASK 3/q3b_variance.png", dpi=120)
plt.show()

**3b(iii) — what the variance distribution reveals.**

The variance is *not* spread evenly across the 3401 frequencies. It is concentrated in a
handful of narrow bands (peaks) separated by broad, low-variance baseline regions — the
variance profile itself looks like a spectrum. Those high-variance bands are exactly the
frequencies where the dominant molecules absorb and where the vials differ most; the
low-variance channels are baseline that is nearly identical across samples and carry
almost no discriminating information.

Concretely, the top ~300 of 3401 features (≈9%) hold ~35% of the total variance, and
~440 features sit above 25% of the peak variance. This concentration means the spectra
are highly redundant and **effectively low-dimensional**: a few linear combinations of the
informative bands capture most of the variation. That is exactly the structure PCA
exploits in 3c, where we expect a small number of large "signal" eigenvalues followed by
a long noise tail.

In [ ]:
#Module purpose. Question 3c: eigendecomposition of C (PCA) and the eigenspectrum.

'''Principal Component Analysis via the covariance matrix.
  Inputs. X_centered, N_tilde (from 3b).
  Method. Form C = (1/N~) X~^T X~ and diagonalise it with numpy.linalg.eigh -- the symmetric
    solver: real eigenvalues, orthonormal eigenvectors, faster and stabler than eig. Sort the
    eigenvalues descending and clip rounding-level negatives to 0 (C is positive semi-definite).
  Analyses. (i) ratio lambda1/lambda2; (ii) the spectrum on linear and log scales; (iii) the
    elbow k* = the largest multiplicative drop between consecutive eigenvalues (k*=8); cumulative
    variance (95% needs 667 components); and a saved ratios table (PNG + CSV).
  Outputs. C, eigvals, eigvecs (feature-space PCs, reused in 3d/3e), k_star; figures
    q3c_spectrum.png, q3c_cvarr.png, q3c_ratios.png and the table q3c_ratios.csv.'''

In [ ]:
#Cell 16 — Question 3c: eigendecomposition of the covariance matrix (PCA)

# Covariance matrix  C = (1/N~) X~^T X~   (M x M = 3401 x 3401).
C = (X_centered.T @ X_centered) / N_tilde

# eigh is for SYMMETRIC matrices: it guarantees real eigenvalues and an
# orthonormal eigenvector basis, and is faster/more stable than eig.
# It returns them in ASCENDING order, so we reverse to largest-first.
w, V = np.linalg.eigh(C)
order   = np.argsort(w)[::-1]
eigvals = np.clip(w[order], 0, None)    # clip tiny negatives caused by round-off
eigvecs = V[:, order]                   # columns = principal directions nu_k (feature space)

# Sanity check: sum of eigenvalues == trace(C) == total variance.
print("sum(eigenvalues) =", round(float(eigvals.sum()), 4),
      " vs  tr(C) =", round(float(C.trace()), 4))

# 3c(i): ratio of the two largest eigenvalues
print(f"3c(i)  lambda1/lambda2 = {eigvals[0] / eigvals[1]:.3f}")

In [ ]:
#Cell 17 — Question 3c(ii)+(iii): eigenvalue spectrum (linear + log) and the elbow

# 3c(iii): locate the elbow = the largest multiplicative drop between
# consecutive eigenvalues in the leading part of the spectrum.
ratios = eigvals[:30] / eigvals[1:31]
k_star = int(np.argmax(ratios)) + 1             # number of "signal" eigenvalues
print(f"3c(iii)  elbow at k* = {k_star}  "
      f"(drop lambda{k_star}/lambda{k_star+1} = {ratios[k_star-1]:.2f})")

top = 50                                         # show the leading spectrum
k = np.arange(1, top + 1)
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
for a, scale in zip(ax, ["linear", "log"]):
    a.plot(k, eigvals[:top], "o-", ms=4, color="steelblue")
    a.axvline(k_star + 0.5, color="red", ls="--", label=f"elbow k* = {k_star}")
    a.set_yscale(scale)
    a.set_xlabel("component index k")
    a.set_ylabel(r"eigenvalue $\lambda_k$")
    a.set_title(f"Eigenvalue spectrum ({scale} scale)")
    a.legend()
plt.tight_layout()
os.makedirs("data/output/TASK 3", exist_ok=True)
plt.savefig("data/output/TASK 3/q3c_spectrum.png", dpi=120)
plt.show()

**3c(iii) — the elbow.** On the log-scale plot the eigenvalues fall steeply for the first
eight components and then flatten into a near-horizontal *noise floor*. The boundary is
sharp: λ₈ ≈ 0.039 but λ₉ ≈ 0.0093, a **4.1× drop** — the largest consecutive drop in the
spectrum — after which every further eigenvalue is ≈ 0.009 and decreases only marginally.
So **k\* = 8**: eight large "signal" eigenvalues followed by a long flat tail of ~900 small
"noise" eigenvalues. This is satisfyingly close to the ~10 dominant molecules named in 3h —
the spectra are mixtures of a handful of chemical components, so only a handful of
independent directions carry real signal.

**3c(iv) — what a large ratio means.** λ₁/λ₂ ≈ 3.5, and λ₁ alone explains ≈ 31% of the total
variance — far more than any other direction. A large ratio means **one direction dominates
the sample-to-sample variation**: the data has a strong principal axis and is effectively
low-dimensional, so projecting onto a few leading eigenvectors loses very little. (It also
means the Power Method of Q2 would converge quickly here, since the spectral gap λ₁ − λ₂ is
wide.)

In [ ]:
#Cell 18 — Question 3c (extra): cumulative variance explained

# Fraction of total variance captured by the top-k principal components.
cumvar = np.cumsum(eigvals) / eigvals.sum()

print(f"signal components (elbow): k* = {k_star}  ->  {100*cumvar[k_star-1]:.1f}% of variance")
for pct in [0.80, 0.90, 0.95, 0.99]:
    k_pct = int(np.searchsorted(cumvar, pct)) + 1
    print(f"{int(pct*100)}% of variance reached at k = {k_pct}")

k95 = int(np.searchsorted(cumvar, 0.95)) + 1

# Plot cumulative variance vs k, marking the elbow and the 95% point.
fig, ax = plt.subplots(figsize=(11, 4.5))
ax.plot(np.arange(1, len(cumvar) + 1), cumvar, color="steelblue")
ax.axhline(0.95, color="gray", ls=":", lw=1)
ax.axvline(k_star, color="red",   ls="--", lw=1,
           label=f"elbow k* = {k_star}  ({100*cumvar[k_star-1]:.0f}% of variance)")
ax.axvline(k95,    color="green", ls="--", lw=1,
           label=f"95% variance at k = {k95}")
ax.set_xlabel("number of components k")
ax.set_ylabel("cumulative variance explained")
ax.set_title("Cumulative variance: 8 signal components reach only ~55%; 95% needs ~670")
ax.set_ylim(0, 1.02)
ax.legend()
plt.tight_layout()
os.makedirs("data/output/TASK 3", exist_ok=True)
plt.savefig("data/output/TASK 3/q3c_cvarr.png", dpi=120)
plt.show()

In [ ]:
#Cell 20 — Question 3c: table of PCA eigenvalue ratios (saved to the q3c outputs)

# For each leading component k: the eigenvalue, the consecutive ratio
# lambda_k / lambda_{k+1} (whose largest value marks the elbow), and the
# per-component and cumulative share of the total variance.
K = 12
tot = eigvals.sum()
rows, ratios_list = [], []
for k in range(K):
    lam = eigvals[k]
    ratio = eigvals[k] / eigvals[k + 1]
    ratios_list.append(ratio)
    rows.append([str(k + 1), f"{lam:.4f}", f"{ratio:.2f}",
                 f"{100 * lam / tot:.1f}%", f"{100 * eigvals[:k + 1].sum() / tot:.1f}%"])
elbow = int(np.argmax(ratios_list))   # the largest consecutive drop = the elbow

fig, ax = plt.subplots(figsize=(8, 4.8)); ax.axis("off")
tbl = ax.table(cellText=rows,
               colLabels=["k", "eigenvalue \u03bbk", "ratio \u03bbk/\u03bbk+1",
                          "variance %", "cumulative %"],
               loc="center", cellLoc="center")
tbl.auto_set_font_size(False); tbl.set_fontsize(10); tbl.scale(1, 1.5)
for j in range(5):
    tbl[elbow + 1, j].set_facecolor("#ffe08a")   # highlight the elbow row
ax.set_title(f"PCA eigenvalue ratios — largest drop \u03bb{elbow+1}/\u03bb{elbow+2} = "
             f"{ratios_list[elbow]:.2f} marks the elbow k* = {elbow+1}")
plt.tight_layout()
os.makedirs("data/output/TASK 3", exist_ok=True)
plt.savefig("data/output/TASK 3/q3c_ratios.png", dpi=120, bbox_inches="tight")
plt.show()

print(f"PCA eigenvalue ratios (top {K}):")
print(f"{'k':>2}  {'lambda':>9}  {'ratio':>6}  {'var%':>6}  {'cum%':>6}")
for r in rows:
    print(f"{r[0]:>2}  {r[1]:>9}  {r[2]:>6}  {r[3]:>6}  {r[4]:>6}")

In [ ]:
#Cell 22 — Question 3c: save the eigenvalue-ratio table (k = 1..9) as a loadable CSV
import csv
tot = eigvals.sum()
K = 9
os.makedirs("data/output/TASK 3", exist_ok=True)
csv_path = "data/output/TASK 3/q3c_ratios.csv"
with open(csv_path, "w", newline="") as f:
    wr = csv.writer(f)
    wr.writerow(["k", "eigenvalue", "ratio_k_over_k+1", "variance_percent", "cumulative_percent"])
    for k in range(K):
        wr.writerow([k + 1,
                     round(float(eigvals[k]), 6),
                     round(float(eigvals[k] / eigvals[k + 1]), 4),
                     round(float(100 * eigvals[k] / tot), 3),
                     round(float(100 * eigvals[:k + 1].sum() / tot), 3)])
print("saved", csv_path)

# verify it loads back
with open(csv_path, newline="") as f:
    for row in csv.reader(f):
        print(row)

**Cumulative variance — why the elbow, not the "95%" rule, sets the dimensionality.**
The 8 signal components (the elbow) capture only ≈ 55.5% of the total variance; reaching 95%
requires **k = 667** components. That gap exists because the variance beyond the elbow is
spread thinly over hundreds of small *noise* eigenvalues, which add up collectively even
though none is individually meaningful. So a "keep 95% of the variance" cut-off would pull in
~660 noise directions and badly **over**estimate the data's dimensionality. The elbow /
spectral-gap criterion (k\* = 8) is the right one here, and it matches the ~10 dominant
molecules of 3h.

In [ ]:
#Module purpose. Question 3d: interpret the leading eigenvector v1.

'''The leading eigenvector (first principal component).
  Inputs. eigvecs, X_centered, S_clean (from 3b/3c).
  Method. Take v1 = eigvecs[:,0] (an M=3401-dimensional vector in feature space) and fix its sign
    so the largest-magnitude loading is positive (eigenvectors are defined only up to a sign).
    Plot its loadings against frequency, then test whether it resembles a real measurement using
    cosine similarity against the centered rows, the raw rows, and the mean spectrum.
  Key finding. v1 is a CONTRAST (mixed-sign loadings concentrated on two molecular bands); it
    matches a centered spectrum at cos ~0.92 but the mean at ~0.03 -- it is a difference-spectrum.
  Outputs. v1; figures q3d_v1.png and q3d_v1_vs_row.png.'''

In [ ]:
#Cell 19 — Question 3d(i): plot the leading eigenvector v1

v1 = eigvecs[:, 0].copy()
# eigenvectors are defined up to a sign; flip so the largest component is positive
if v1[np.argmax(np.abs(v1))] < 0:
    v1 = -v1

# which frequencies carry the most weight in v1?
top = np.argsort(np.abs(v1))[::-1][:10]
print("most heavily weighted features (j, v1_j):",
      [(int(j), round(float(v1[j]), 3)) for j in top])
print("fraction of negative components:", round(float((v1 < 0).mean()), 3))

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(v1, lw=0.6, color="darkgreen")
ax.axhline(0, color="k", lw=0.5)
ax.set_xlabel("feature index j  (frequency channel)")
ax.set_ylabel(r"$\nu_1$ component")
ax.set_title(r"Leading eigenvector $\nu_1$  (direction of greatest variation)")
plt.tight_layout()
os.makedirs("data/output/TASK 3", exist_ok=True)
plt.savefig("data/output/TASK 3/q3d_v1.png", dpi=120)
plt.show()

**3d(i) — interpreting ν₁.** The leading eigenvector is *not* flat: its weight is concentrated
in two narrow frequency bands — around **j ≈ 900–960** (the largest components: j = 910, 922,
942, 909, …) and **j ≈ 3150–3260** (j = 3150, 3254). These are exactly the high-variance
molecular bands seen in 3b. ν₁ also has both **positive and negative lobes** (~27 % of its
components are negative), so it is a *contrast* direction: it separates samples by the relative
balance of absorption between these bands, not by overall brightness. The single summary number
z = x·ν₁ therefore measures "how strongly a vial absorbs in the 900–960 / 3150–3260 bands versus
the rest" — the chemical contrast that most distinguishes the concoctions.

In [ ]:
#Cell 21 — Question 3d(ii): does v1 resemble a measurement?

def cosines(M, v):                      # cosine similarity of v with every row of M
    """Cosine similarity between v and every row of M: returns an array whose i-th entry is (M[i].v)/(||M[i]|| ||v||). Scale-invariant, so it compares direction/shape rather than magnitude."""
    return (M @ v) / (np.linalg.norm(M, axis=1) * np.linalg.norm(v))

cos_centered = cosines(X_centered, v1)              # vs mean-centered rows (rows of X~)
i_best = int(np.argmax(np.abs(cos_centered)))
mean_spec = S_clean.mean(0)
print(f"best match: centered row #{i_best}, cos = {cos_centered[i_best]:.3f}")
print(f"cos(v1, raw row #{i_best})  = {cosines(S_clean, v1)[i_best]:.3f}")
print(f"cos(v1, mean spectrum)     = "
      f"{(v1 @ mean_spec) / (np.linalg.norm(v1) * np.linalg.norm(mean_spec)):.3f}")

# overlay v1 with the best-matching centered measurement (both unit-normalised)
row = X_centered[i_best] / np.linalg.norm(X_centered[i_best])
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(v1,  lw=0.6, color="darkgreen", label=r"$\nu_1$")
ax.plot(row, lw=0.6, color="orange", alpha=0.8,
        label=f"centered measurement #{i_best} (unit norm)")
ax.set_xlabel("feature index j"); ax.set_ylabel("amplitude")
ax.set_title(rf"$\nu_1$ vs best-matching centered spectrum  (cos = {cos_centered[i_best]:.2f})")
ax.legend()
plt.tight_layout()
plt.savefig("data/output/TASK 3/q3d_v1_vs_row.png", dpi=120)
plt.show()

**3d(ii) — does ν₁ resemble a measurement?** Yes — but a *mean-centered* one. ν₁ matches the
centered spectrum of row #193 with **cosine ≈ 0.92** (overlay above), so the dominant axis of
variation is essentially the deviation-from-average signature of one particular concoction. By
contrast it matches the *raw* (un-centered) spectra only moderately (cos ≈ 0.67) and is
essentially **orthogonal to the mean spectrum** (cos ≈ 0.03). That is exactly what mean-centering
(3b) buys us: the raw spectra are dominated by the shared average spectrum, but ν₁ — living in the
centered space by construction — ignores that common part and captures how the most-deviated
concoction differs from the mean. So ν₁ looks like a real *difference*-spectrum of the data, not
like the average spectrum every vial shares.

In [ ]:
#Module purpose. Question 3e: discover clusters in covariance (feature) space.

'''Clustering via the top-4 covariance eigenvectors.
  Inputs. X_centered, eigvecs (from 3c).
  Method. Project each vial onto the top-4 principal components, Z = X~ V4 (924 x 4), and show all
    6 pairwise 2-D scatter plots with small, semi-transparent points so dense groups stay visible.
    Count clusters with a numpy-only leader-clustering routine whose radius is a multiple of the
    median nearest-neighbour spacing, and confirm the count is stable across that radius.
  Outputs. Z (covariance-space scores); figure q3e_covariance_scatter.png; cluster count = 25.'''

In [ ]:
#Cell 23 — Question 3e(i): project onto the top 4 PCs and scatter all 6 pairs

# Z = X~ V4 : each cleaned vial summarised by its scores on the 4 leading
# eigenvectors (a 4-D compressed coordinate). Shape (924, 4).
Z = X_centered @ eigvecs[:, :4]

pairs = [(0, 1), (0, 2), (0, 3), (1, 2), (1, 3), (2, 3)]
fig, ax = plt.subplots(2, 3, figsize=(15, 9))
for a, (p, q) in zip(ax.ravel(), pairs):
    a.scatter(Z[:, p], Z[:, q], s=10, alpha=0.4)      # small, semi-transparent points
    a.set_xlabel(f"PC{p+1}"); a.set_ylabel(f"PC{q+1}")
fig.suptitle("Covariance-space projection: all 6 pairwise scatter plots of the top-4 PCs")
plt.tight_layout()
os.makedirs("data/output/TASK 3", exist_ok=True)
plt.savefig("data/output/TASK 3/q3e_covariance_scatter.png", dpi=110)
plt.show()

In [ ]:
#Cell 24 — Question 3e(ii): how many distinct clusters?

def count_clusters(P, r):
    """Leader clustering (numpy-only, no sklearn): scan the points P and start a new cluster centre whenever a point lies farther than radius r from every existing centre. Returns the number of clusters found."""
    # leader clustering (numpy-only): new cluster when a point is farther than r
    # from every existing cluster centre
    centres = [P[0]]
    for p in P[1:]:
        if np.sqrt(((np.array(centres) - p) ** 2).sum(1)).min() > r:
            centres.append(p)
    return len(centres)

D = np.sqrt(((Z[:, None] - Z[None]) ** 2).sum(-1)); np.fill_diagonal(D, np.inf)
r = 8 * np.median(D.min(1))                 # radius ~ 8x the typical neighbour spacing
print(f"distinct clusters (covariance space): {count_clusters(Z, r)}")

**3e(ii) — counting the clusters.** The six panels show many tight, well-separated blobs. A simple
distance-based count (leader clustering, stable for a radius of 6–12× the typical neighbour spacing)
gives **25 clusters**. Each blob is a group of vials with almost identical spectra — i.e. one
**concoction**. So the 924 cleaned vials fall into about 25 distinct recipes, comfortably under the
"fewer than 100 unique samples" stated in 3h. The clean separation into a small number of groups is
the visible payoff of the low-dimensional structure found in 3c: projecting onto just 4 of 3401
directions is enough to resolve the recipes.

In [ ]:
#Module purpose. Question 3f: discover clusters in Gram (sample) space.

'''Clustering via the top-4 Gram eigenvectors.
  Inputs. X_centered, N_tilde, eigvals (from 3b/3c).
  Method. Form the Gram matrix G = (1/N~) X~ X~^T (924 x 924, sample space) and diagonalise it. G
    shares the nonzero eigenvalues of C (verified in-cell). Its eigenvectors ARE the sample
    coordinates -- no projection of X~ is needed; we scale them by sqrt(N~*lambda) so the units
    match 3e. Show the same 6 pairwise scatter plots and count clusters the same way.
  Outputs. gvals, gvecs, Zg (Gram-space scores); figure q3f_gram_scatter.png; cluster count = 25.'''

In [ ]:
#Cell 25 — Question 3f(i)+(ii): cluster via the Gram matrix G = (1/N~) X~ X~^T

# The Gram matrix lives in SAMPLE space (N~ x N~); entry G_ij is the similarity
# between vials i and j. It shares the nonzero eigenvalues of the covariance matrix C.
G = (X_centered @ X_centered.T) / N_tilde
wG, UG = np.linalg.eigh(G)
order = np.argsort(wG)[::-1]
gvals = np.clip(wG[order], 0, None)
gvecs = UG[:, order]                       # columns: N~-dimensional sample-space eigenvectors

print("Gram eigenvalues (top 8):", np.round(gvals[:8], 4))
print("match covariance eigenvalues:", np.allclose(gvals[:8], eigvals[:8], atol=1e-6))

# A Gram eigenvector's components ARE the sample coordinates (no projection of X~
# needed). We scale by sqrt(N~ * lambda) so the units match the covariance-space
# scores of 3e, i.e. U*sqrt(N~ lambda) = X~ V.
Zg = gvecs[:, :4] * np.sqrt(N_tilde * gvals[:4])

pairs = [(0, 1), (0, 2), (0, 3), (1, 2), (1, 3), (2, 3)]
fig, ax = plt.subplots(2, 3, figsize=(15, 9))
for a, (p, q) in zip(ax.ravel(), pairs):
    a.scatter(Zg[:, p], Zg[:, q], s=10, alpha=0.4, color="seagreen")
    a.set_xlabel(f"Gram PC{p+1}"); a.set_ylabel(f"Gram PC{q+1}")
fig.suptitle("Gram-space projection: 6 pairwise scatter plots of the top-4 Gram eigenvectors")
plt.tight_layout()
os.makedirs("data/output/TASK 3", exist_ok=True)
plt.savefig("data/output/TASK 3/q3f_gram_scatter.png", dpi=110)
plt.show()

In [ ]:
#Cell 26 — Question 3f(iii): how many clusters in Gram space?

def count_clusters(P, r):
    """Leader clustering (numpy-only, no sklearn): scan the points P and start a new cluster centre whenever a point lies farther than radius r from every existing centre. Returns the number of clusters found."""
    centres = [P[0]]
    for p in P[1:]:
        if np.sqrt(((np.array(centres) - p) ** 2).sum(1)).min() > r:
            centres.append(p)
    return len(centres)

D = np.sqrt(((Zg[:, None] - Zg[None]) ** 2).sum(-1)); np.fill_diagonal(D, np.inf)
r = 8 * np.median(D.min(1))                 # radius ~ 8x the typical neighbour spacing
print(f"distinct clusters (Gram space): {count_clusters(Zg, r)}")

**3f(iii) — clusters via the Gram matrix.** The Gram matrix G = (1/Ñ)·X~ X~^T lives in *sample* space (924×924); its eigenvectors are already Ñ-dimensional, so each component is directly the coordinate of one vial — no projection of X~ is needed. G shares the nonzero eigenvalues of the covariance matrix C (verified above), exactly as Question 2d predicts. Counting with the same leader-clustering rule gives **25 clusters** — identical to 3e and stable across radii 6–12× the median neighbour spacing. The six Gram-space scatter plots reproduce those of 3e up to per-axis sign flips (eigenvectors are defined only up to a sign): the two matrices are two views of the same structure. The detailed comparison is the subject of 3g.

In [ ]:
#Module purpose. Question 3g: compare the covariance (3e) and Gram (3f) clusterings.

'''Comparing the two clustering approaches.
  Inputs. Z (3e covariance scores) and Zg (3f Gram scores).
  Method. Correlate the two coordinate sets component by component. Because both equal U*Sigma in
    the SVD X~ = U Sigma V^T, the matching components are perfectly correlated (|corr| = 1) and the
    cross-correlations are 0 -- so the two scatter plots are identical up to a per-axis sign and
    must yield the same cluster count. The accompanying markdown then explains what cluster
    membership MEANS in feature space versus sample space.
  Outputs. printed correlation matrix (diagonal = +/-1, off-diagonal = 0); figure q3g_compare.png.'''

In [ ]:
#Cell 27 — Question 3g(i): are the 3e (covariance) and 3f (Gram) clusterings the same?

# Both 3e and 3f returned 25 clusters. The reason is that the two coordinate sets are the
# SAME matrix up to a per-axis sign: via the SVD X~ = U S V^T, the covariance scores are
# Z = X~ V = U S, while the Gram coordinates are Zg = U * sqrt(N~ * lambda) = U S. We confirm
# this by correlating the 3e scores Z with the 3f scores Zg, component by component.
def _corr(a, b):
    """Pearson correlation coefficient between two 1-D arrays a and b (mean-centred dot product over the product of their norms); returns a float in [-1, 1]."""
    a = a - a.mean(); b = b - b.mean()
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)))

M = np.array([[_corr(Z[:, i], Zg[:, j]) for j in range(4)] for i in range(4)])
np.set_printoptions(precision=3, suppress=True)
print("correlation matrix (rows = 3e PCs, cols = 3f PCs):")
print(M)
print("per-matching-component |correlation|:", np.round(np.abs(np.diag(M)), 4))
print("max off-diagonal |correlation|      :", round(float(np.abs(M - np.diag(np.diag(M))).max()), 4))
print("coordinates equal up to per-axis sign?", bool(np.allclose(np.abs(Z), np.abs(Zg), atol=1e-6)))

## Question 3g — comparing the covariance and Gram clusterings

**3g(i) — Do we find the same number of clusters? Yes (25 in both), and necessarily so.**

The mathematical connection: if ν is an eigenvector of C = (1/Ñ)·X~^T X~ with eigenvalue λ, then X~ ν is an eigenvector of G = (1/Ñ)·X~ X~^T with the same eigenvalue λ (substitute and check). So C and G share their nonzero eigenvalues — verified exactly above (top-8 identical).

More strongly, write the SVD X~ = U Σ V^T. The covariance eigenvectors are the columns of V and the covariance-space scores are Z = X~ V = U Σ; the Gram eigenvectors are the columns of U and, scaled by √(Ñ·λ) = σ, the Gram coordinates are Zg = U·σ = U Σ. **Both coordinate sets are the same matrix U Σ** — identical up to the arbitrary sign of each eigenvector. The cell above confirms it: every matching component is perfectly correlated (|corr| = 1.000), all cross-correlations are 0, and |Z| = |Zg| to machine precision (here PC1 and PC4 came out sign-flipped). Because the two point clouds are the same up to reflecting individual axes, the clusters — and their number — must be identical. (In general the two pictures can differ slightly when one matrix is better conditioned numerically, but here the structure is clean enough that they agree exactly.)

**3g(ii) — What it means for a vial to belong to a cluster, in each space.**

*Covariance space (3e):* a vial's coordinates are the projections of its centred spectrum onto the top-4 **feature-space** eigenvectors, z_i = (x~_i · ν_1, …, x~_i · ν_4). Each coordinate measures how strongly the vial expresses one dominant pattern of spectral variation — a fixed weighted combination of frequencies (e.g. ν_1's contrast between molecular bands). Two vials are close when they express these dominant chemical patterns to the same degree, i.e. their spectra are similar mixtures along the directions in which samples differ most. A cluster is a set of vials sharing that fingerprint.

*Gram space (3f):* a vial's coordinates are the components of the top-4 **sample-space** eigenvectors of G. Since G_ij is the dot-product similarity between vials i and j, each Gram eigenvector is a pattern over the 924 vials, and a vial's coordinate is its loading in that pattern — a compressed summary of *how similar this vial is to every other vial*. Two vials are close when they have the same similarity-fingerprint to the rest of the dataset.

*Same conclusion, two viewpoints:* covariance space asks "which mixture of frequencies does this vial express?"; Gram space asks "which other vials does this one resemble?" Both place vials with near-identical spectra at the same point, so proximity in either coordinate system means the underlying vials hold the **same chemical concoction** — which is why the two routes agree on 25 clusters.

In [ ]:
#Cell 28 — Question 3g: figure comparing the 3e (covariance) and 3f (Gram) clusterings

# Two complementary proofs that 3e and 3f are the same projection up to per-axis sign:
#   (left)  algebraic — the 4x4 |correlation| matrix of the 3e scores Z vs the 3f
#           scores Zg: =1 on the diagonal (each PC perfectly matched), 0 off-diagonal.
#   (right) visual — the same 6 PC-pair scatters as 3e/3f, overlaid. The Gram scores
#           are sign-flipped per axis to match 3e, after which every point coincides,
#           so the two routes resolve the identical 25 clusters.
def _corr(a, b):
    """Pearson correlation coefficient between two 1-D arrays a and b (mean-centred dot product over the product of their norms); returns a float in [-1, 1]."""
    a = a - a.mean(); b = b - b.mean()
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)))

Mcorr = np.array([[_corr(Z[:, i], Zg[:, j]) for j in range(4)] for i in range(4)])
signs = np.sign(np.diag(Mcorr))          # +-1 sign of each matched component
Zg_al = Zg * signs                        # flip each Gram axis to match 3e's sign
print("max |Z - Zg(sign-aligned)| =", float(np.abs(Z - Zg_al).max()))   # ~1e-14

fig = plt.figure(figsize=(16, 8))
gs = fig.add_gridspec(2, 4, width_ratios=[1.15, 1, 1, 1])

# left: |correlation| heatmap (algebraic proof)
axh = fig.add_subplot(gs[:, 0])
im = axh.imshow(np.abs(Mcorr), cmap="viridis", vmin=0, vmax=1)
axh.set_xticks(range(4)); axh.set_yticks(range(4))
axh.set_xticklabels([f"3f PC{j+1}" for j in range(4)], rotation=45, ha="right")
axh.set_yticklabels([f"3e PC{i+1}" for i in range(4)])
for i in range(4):
    for j in range(4):
        v = abs(Mcorr[i, j])
        axh.text(j, i, f"{v:.2f}", ha="center", va="center",
                 color="white" if v < 0.5 else "black", fontsize=10)
axh.set_title("|correlation| of 3e vs 3f scores\n(=1 on diagonal, 0 off)")
fig.colorbar(im, ax=axh, fraction=0.046, pad=0.04)

# right: 6 PC-pair overlays (visual proof)
pairs = [(0, 1), (0, 2), (0, 3), (1, 2), (1, 3), (2, 3)]
cells = [gs[0, 1], gs[0, 2], gs[0, 3], gs[1, 1], gs[1, 2], gs[1, 3]]
for cell, (p, q) in zip(cells, pairs):
    a = fig.add_subplot(cell)
    a.scatter(Z[:, p], Z[:, q], s=42, facecolors="none", edgecolors="steelblue",
              linewidths=0.8, label="3e (covariance)")
    a.scatter(Zg_al[:, p], Zg_al[:, q], s=6, color="darkorange", alpha=0.8,
              label="3f (Gram, sign-aligned)")
    a.set_xlabel(f"PC{p+1}"); a.set_ylabel(f"PC{q+1}")
    if (p, q) == (0, 1):
        a.legend(fontsize=8, loc="best")

flips = ", ".join(f"PC{k+1}" for k in range(4) if signs[k] < 0) or "none"
fig.suptitle("Q3g — covariance (3e) and Gram (3f) projections are the same point cloud "
             f"up to per-axis sign (flipped: {flips})  ->  identical 25 clusters",
             fontsize=13)
plt.tight_layout(rect=[0, 0, 1, 0.96])
os.makedirs("data/output/TASK 3", exist_ok=True)
plt.savefig("data/output/TASK 3/q3g_compare.png", dpi=120, bbox_inches="tight")
plt.show()

## Question 3h — The forgetful taste tester: how many unique samples?

The vials come from a blind taste test (fermented caffeinated drinks: kombucha, cocoa drinks, coffee liqueurs, Chinese tea) whose ground-truth labels were lost. We are told there were **fewer than 100** unique samples (one per vial) and **ten dominant molecules** (water, ethanol, methanol, caffeine, propionic acid, formic acid, lactic acid, glycerol, acetaldehyde, acetic acid) that the infrared spectroscopy was designed to fingerprint.

**(i) How many unique samples? ≈ 25.** After removing the 40 empty vials and the 36 corrupted high-norm vials, the 924 genuine spectra fall into **25** tight clusters (parts 3e and 3f), each cluster a set of near-identical spectra — i.e. one recipe. The count is *stable* rather than fitted: it is invariant for clustering radii across 6–12× the median nearest-neighbour spacing, and the covariance and Gram routes return the same 25 (3g), which by the Q2(d) duality is a genuine cross-check, not a coincidence. It sits comfortably under the stated bound of 100. The ten dominant molecules manifest as the **k\* = 8** signal eigenvalues of 3c: each spectrum is a non-negative mixture of about ten absorption signatures, so the mean-centred data spans only ~8–10 independent directions. (The 36 high-norm vials are a measurement fault from 3a, not a 26th recipe.)

**(ii) Which earlier parts were useful?** Almost all of them. Cleaning (3a) was a precondition — the empties and outliers would otherwise have manufactured spurious clusters and inflated the variance. Mean-centring and the covariance matrix (3b), then its eigenspectrum (3c), compressed 3401 noisy features to the 8 that carry signal. The projection and clustering of 3e produced the count, the Gram route (3f) reproduced it, and the comparison (3g) turned that agreement into a cross-check, resting on the XᵀX / XXᵀ duality proved in Q2(d). Even Q1 supplies the governing intuition: the leading eigenvector of the right matrix carries the structure a system settles into. The figure of **25 unique samples** is the consistent reading of the whole pipeline.

In [ ]:
#Cell 29 — Question 3h: label the 25 clusters and show the cleaned vials coloured by group

'''The 25 unique samples, visualised.
  Inputs. Z (924 x 4 covariance scores from 3e).
  Method. Re-run the SAME leader clustering as 3e, but this time record a label for every vial:
    a vial farther than the radius r from all existing centres starts a new cluster, otherwise it
    joins the nearest existing centre. The radius r is the same 8x median nearest-neighbour spacing
    used for the count in 3e, so the labelling is consistent with the cluster count of 25.
  Outputs. printed cluster sizes; figure q3h_clusters.png -- the 924 cleaned vials resolving into
    ~25 groups in the leading covariance-score plane. (We deliberately omit a replicate-count bar
    chart; the coloured scatter already shows the grouping.)'''

def cluster_labels(P, r):
    """Leader clustering that returns a label per point (not just the count, as count_clusters did).
    A point starts a new cluster when it is farther than r from every existing centre; otherwise it
    joins the nearest existing centre. Returns a length-len(P) array of integer cluster labels."""
    centres = [P[0]]
    labels = [0]
    for i in range(1, len(P)):
        p = P[i]
        dists = np.sqrt(((np.array(centres) - p) ** 2).sum(1))
        nearest = int(np.argmin(dists))
        if dists[nearest] > r:
            centres.append(p)
            labels.append(len(centres) - 1)
        else:
            labels.append(nearest)
    return np.array(labels)

# Use the same radius as the 3e cluster count.
D = np.sqrt(((Z[:, None] - Z[None]) ** 2).sum(-1)); np.fill_diagonal(D, np.inf)
r = 8 * np.median(D.min(1))

labels = cluster_labels(Z, r)
n_clusters = int(labels.max()) + 1
sizes = np.array([int((labels == c).sum()) for c in range(n_clusters)])
print(f"clusters found: {n_clusters}")
print(f"cluster sizes : min {sizes.min()}, max {sizes.max()}, mean {sizes.mean():.1f}, total {int(sizes.sum())}")
print(f"size counts   : {dict(zip(*np.unique(sizes, return_counts=True)))}")

# Scatter the leading two covariance scores, coloured by cluster.
fig, ax = plt.subplots(figsize=(7, 5.5))
ax.scatter(Z[:, 0], Z[:, 1], c=labels, cmap="tab20", s=14)
ax.set_xlabel("PC1"); ax.set_ylabel("PC2")
ax.set_title(f"The {n_clusters} unique samples: {Z.shape[0]} cleaned vials coloured by cluster")
plt.tight_layout()
os.makedirs("data/output/TASK 3", exist_ok=True)
plt.savefig("data/output/TASK 3/q3h_clusters.png", dpi=120)
plt.show()
